# Crop-type mask — viewer

Read-only tour of everything the pipeline produced, per country:

| | Ethiopia | Kenya | Uganda | Tanzania | Rwanda | Burundi | Somalia | South Sudan | Sudan | Eritrea | Djibouti |
|---|---|---|---|---|---|---|---|---|---|---|---|
| Profile / folder | `Ethiopia` | `Kenya` | `Uganda` | `Tanzania` | `Rwanda_SPAM2020` | `Burundi_SPAM2020` | `Somalia_SPAM2020` | `South_Sudan` | `Sudan` | `Eritrea` | `Djibouti` |
| Crops | maize, wheat, sorghum, teff | maize, wheat, sorghum | maize, sorghum | maize, wheat, sorghum | maize, wheat, sorghum | maize, wheat, sorghum | maize, sorghum | maize, sorghum | sorghum, millet, wheat | sorghum, millet | cropland only |
| Calibrated to | Meher zones 2011–16 | counties 2015–20 | districts 2008/09 | regions 2010–14 (cleaned) | districts 2010–17 | provinces 2013–14 | districts 2015–19 | — (uncalibrated) | states 2015–20 | — (uncalibrated) | — |
| Validation | LSMS-ISA EAs | One Acre Fund plots | LSMS-ISA + Dalberg | LSMS-ISA EAs | Niche trial fields | One Acre Fund (5 km) | held-out 2020–24 stats | none | held-out 2021–23 stats | none | none |

Rwanda, Burundi and Somalia releases use SPAM 2020 instead of the four-edition median (see their METHODOLOGY Section 3.2); the median versions are in `Rwanda/`, `Burundi/` and `Somalia/` for comparison. Sudan also uses SPAM 2020, and votes cropland on the 100 m grid inside the SPAM crop footprint (1.88 M km2 is otherwise too slow). South Sudan is uncalibrated (no usable statistics) and uses a looser cropland rule (any one mapping lineage, METHODOLOGY Section 3.1). Eritrea is absent from HarvestStat: uncalibrated, GADM regions. Djibouti is a cropland-only layer (SPAM 2020 has 47 ha of sorghum), kept separate; its usable layer is "any one lineage" (vote >= 25 %).

**This notebook only displays.** To (re)build anything, use `crop_type_mask_colab.ipynb`.

## 0 — Setup (Drive + Earth Engine)

### Stage 0 · Setup

**What this notebook is.** A read-only tour of a finished country. It builds nothing and exports
nothing, so it is safe to hand to someone who should look at the product but not rebuild it. To
rebuild, use `crop_type_mask_colab.ipynb`.

**What you set.** `PIPE_DIR`, the Earth Engine project, and `COUNTRY`. Eleven profiles exist; note that
Rwanda, Burundi and Somalia carry the `_SPAM2020` suffix, and that Djibouti is cropland only.

**Expected output.** `Kenya KE crops: ['maize', 'wheat', 'sorghum'] -> .../Kenya`.

In [ ]:
!pip -q install geemap rasterio openpyxl
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob
PIPE_DIR = "/content/drive/MyDrive/planting_pipeline"      # adjust if your folder is elsewhere
os.environ["CTM_EE_PROJECT"] = "indigo-proxy-484220-q8"     # your Earth Engine cloud project
sys.path.insert(0, PIPE_DIR)

import ee
ee.Authenticate()
from crop_type_mask import config as C, ee_utils as U, run_mask as R
U.init(ee)

COUNTRY = "Kenya"            # "Ethiopia" | "Kenya" | "Uganda" | "Tanzania" | "Rwanda_SPAM2020" | "Burundi_SPAM2020" | "Somalia_SPAM2020" | "South_Sudan" | "Sudan" | "Eritrea" | "Djibouti"
R.use_country(COUNTRY)
print(C.COUNTRY, C.ISO2, "crops:", C.CROP_NAMES, "->", C.COUNTRY_DIR)

## 1 — What exists for this country
Earth Engine assets and the files on disk. Anything missing is listed, so a half-finished country is obvious.

### Stage 1 · What exists for this country

**What this stage does.** Lists the six Earth Engine assets a finished country should have, then the
GeoTIFFs and documents on disk. A half-finished country is obvious from the `exists` column.

**Expected output.** Six `True` values for a finished country, five GeoTIFFs
(`crop_fraction`, `crop_mask`, `crop_confidence`, `crop_cropland`, `pixel_area_ha`), a `METHODOLOGY.md`
and `.docx`, a statistics workbook and a set of CSVs.

**Exceptions that are not faults.** Djibouti has cropland only, so there is no fraction, mask or
confidence file and no methodology document, only a README. South Sudan, Eritrea and Djibouti have no
calibration unit asset, because they were never calibrated.

In [ ]:
import pandas as pd
assets = {"stable cropland": R.CROPLAND, "climate": f"climate_{C.ISO2}_250m",
          "allocation (prior)": R.PRIOR, "FINAL product": R.PRODUCT,
          "SPAM cells": f"spam_cells_{C.ISO2}", "calibration units": f"zones_{C.ISO2}"}
rows = [{"item": k, "asset": v, "exists": U.asset_exists(ee, R.a(v))} for k, v in assets.items()]
display(pd.DataFrame(rows))

print("\nGeoTIFFs:")
for f in sorted(glob.glob(f"{C.OUT_DIR}/*.tif")):
    print(f"  {os.path.basename(f):38s} {os.path.getsize(f)/1e6:8.1f} MB")
print("\nDocuments and tables:")
for f in sorted(glob.glob(f"{C.COUNTRY_DIR}/METHODOLOGY.*") + glob.glob(f"{C.OUT_DIR}/*.xlsx") + glob.glob(f"{C.OUT_DIR}/*.csv")):
    print("  ", os.path.relpath(f, C.COUNTRY_DIR))

## 2 — Interactive map
Every layer of the final product plus the evidence behind it. Use the layer control (top right) to toggle.

### Stage 2 · The map

**Layers, in the order they answer questions.**

1. **Cropland vote** and **agreement**: the evidence base. How many lineages saw cropland here, and did
   they agree?
2. **Crop fraction** per crop: the product. Percentage of each 100 m cell.
3. **Dominant crop**: which crop wins each cell, maize 1, wheat 2, sorghum 3, teff 4, millet 5.
4. **Confidence**: 0 to 100, from the cropland vote, SPAM presence, how far calibration had to move the
   pixel, and whether anything independent confirmed it.

**What a good country looks like.** Crop fraction concentrated in the known farming zones, dominant
crop following the agro-ecology rather than administrative boundaries, and confidence highest where the
vote was unanimous. Administrative boundaries visible in the **fraction** layer are expected, because
calibration is applied per reporting unit. Boundaries visible in the **cropland vote** are not, and
point at a source-data seam.

In [ ]:
import geemap
CENTER = {"Ethiopia": [9.5, 39.5], "Kenya": [0.3, 37.8], "Uganda": [1.4, 32.4], "Tanzania": [-6.4, 34.9],
          "Rwanda_SPAM2020": [-1.95, 29.9], "Rwanda": [-1.95, 29.9], "Burundi_SPAM2020": [-3.4, 29.9], "Burundi": [-3.4, 29.9],
          "Somalia_SPAM2020": [4.5, 45.3], "Somalia": [4.5, 45.3], "South_Sudan": [7.5, 30.5],
          "Sudan": [13.5, 31.5], "Eritrea": [15.2, 38.6], "Djibouti": [11.8, 42.6]}[COUNTRY]
COL = {"maize": "2a78d6", "wheat": "eb6834", "sorghum": "1baf7a", "teff": "4a3aa7", "millet": "b5195a"}
LIGHT = "f3f2ee"

m = geemap.Map(center=CENTER, zoom=9 if COUNTRY.startswith(("Rwanda", "Burundi")) else 8 if COUNTRY == "Djibouti" else 6)
m.add_basemap("HYBRID")

cl = ee.Image(R.a(R.CROPLAND))
m.addLayer(cl.select("vote_pct"), {"min": 0, "max": 100, "palette": [LIGHT, "0b0b0b"]}, "cropland: mean vote %", False)
m.addLayer(cl.select("agree_pct"), {"min": 0, "max": 100, "palette": ["d55e00", LIGHT, "009e73"]}, "cropland: source agreement %", False)
m.addLayer(cl.select("cl_pct").selfMask(), {"min": 0, "max": 100, "palette": [LIGHT, "8c6d31", "3d2b0f"]}, "STABLE CROPLAND % of cell", COUNTRY != "Djibouti")
if COUNTRY == "Djibouti":   # majority vote leaves ~50 ha; any one lineage is the usable layer
    m.addLayer(cl.select("vote_pct").gte(25).selfMask(), {"palette": ["b5651d"]}, "CROPLAND: any one lineage (vote >= 25 %)", True)

if U.asset_exists(ee, R.a(R.PRODUCT)):
    p = ee.Image(R.a(R.PRODUCT))
    for c in C.CROP_NAMES:
        m.addLayer(p.select(f"conf_{c}").updateMask(p.select(f"mask_{c}")), {"min": 30, "max": 90, "palette": [LIGHT, "0b0b0b"]}, f"confidence: {c}", False)
        m.addLayer(p.select(f"mask_{c}").selfMask(), {"palette": [COL[c]]}, f"mask: {c}", False)
        m.addLayer(p.select(f"frac_{c}").selfMask(), {"min": 0, "max": 40, "palette": [LIGHT, COL[c]]}, f"fraction %: {c}", c == C.CROP_NAMES[-1])
    m.addLayer(p.select("dominant").selfMask(), {"min": 1, "max": 5,
               "palette": [COL["maize"], COL["wheat"], COL["sorghum"], COL["teff"], COL["millet"]]}, "DOMINANT crop", True)

m.addLayer(ee.Image().byte().paint(ee.FeatureCollection(R.a(f"zones_{C.ISO2}")), 1, 1), {"palette": ["ffffff"]}, "calibration units", True)
m.add_inspector()          # click the map to read every band
m

## 3 — Headline numbers
National totals against the official statistics, and the validation.

### Stage 3 · Headline numbers

**What this stage does.** Prints the national totals against the official statistics, and the
validation tables.

**How to read `final_national_summary`.** `mapped_ha` is the area the product holds, from the crop
fraction. `target_ha` is what the statistics report. `mask_ha` is the binary mask, which is always
larger and is **not an area**. `conf` is the mean confidence inside the mask.

**Expected values.** Ethiopia places 96 to 100 % of its reported area, Kenya 95 to 105 %, Uganda 83 to
97 %, Tanzania 82 to 92 %, Rwanda 89 to 98 %, Burundi 70 to 98 %, Somalia 72 to 88 %, Sudan 69 to 84 %.
South Sudan, Eritrea and Djibouti have no target, because no statistics exist; their `target_ha` is
blank and their areas are a lower bound, not an estimate.

**Validation.** `validation_zones` is agreement with the statistics, which the product was fitted to.
The `validation_*_metrics` files are the independent tests and are the ones that count. Read the AUC
against the ingredient baselines printed beside it, not on its own.

In [ ]:
ST = f"{C.OUT_DIR}/stats"
def show(name, **kw):
    f = f"{ST}/{name}_{C.ISO2}.csv"
    if os.path.exists(f):
        print(f"\n--- {name} ---")
        display(pd.read_csv(f).round(3))
show("final_national_summary")
show("cropland_by_region")          # Djibouti (cropland-only)
show("cropland_by_vote_threshold")
show("validation_zones")
for v in ["validation_lsms_metrics", "validation_oaf_metrics", "validation_dalberg_metrics"]:
    show(v)

## 4 — All figures

### Stage 4 · Every figure

Thirteen or fourteen figures per country, numbered to match the methodology document. The fastest read
is figure 6 for the cropland disagreement, figure 8 for where calibration hit its limits, and figure 11
for whether the dominant crop follows the agro-ecology.

In [ ]:
from IPython.display import Image, display
figs = sorted(glob.glob(f"{C.OUT_DIR}/figures/*.png"))
print(len(figs), "figures")
for f in figs:
    print("\n" + os.path.basename(f))
    display(Image(f, width=1000))

## 5 — The statistics workbook (every table used)

### Stage 5 · The statistics workbook

Every table behind the figures, one sheet each: SPAM stability, cropland by source-year, the
calibration targets, the per-unit before and after, the distribution of $k$, the national summary, and
the validation. This is the file to send when someone asks for the numbers rather than the maps.

In [ ]:
xl = f"{C.OUT_DIR}/crop_type_mask_statistics_{C.ISO2}.xlsx"
book = pd.read_excel(xl, sheet_name=None) if os.path.exists(xl) else {}
print(xl, "|", len(book), "sheets")
for name, df in book.items():
    print(f"\n--- {name} ({len(df)} rows) ---")
    display(df.head(12))

## 6 — Compare countries side by side
National totals and how far the statistics had to move each prior.

### Stage 6 · All countries side by side

**What this stage does.** Loops every profile and concatenates the national summaries, so the eleven
products can be compared in one table.

**What the comparison shows.** The share of reported area placed falls as you move from countries with
recent statistics and dense cropland (Ethiopia, Kenya) to countries with old statistics or arid,
shifting cultivation (Sudan millet at 69 %). The pattern is a property of the **reference data**, not
of the method; the method is identical in all eleven.

In [ ]:
rows = []
for country in ["Ethiopia", "Kenya", "Uganda", "Tanzania", "Rwanda_SPAM2020", "Burundi_SPAM2020", "Somalia_SPAM2020", "South_Sudan", "Sudan", "Eritrea"]:
    R.use_country(country)
    f = f"{C.OUT_DIR}/stats/final_national_summary_{C.ISO2}.csv"
    if os.path.exists(f):
        d = pd.read_csv(f); d.insert(0, "country", country); rows.append(d)
R.use_country(COUNTRY)
display(pd.concat(rows, ignore_index=True).round(3) if rows else "no country finished yet")

## 7 — Read the GeoTIFFs directly (optional)
The same numbers as the Earth Engine asset, as local files.

### Stage 7 · Read the GeoTIFFs directly

**What this stage does.** Opens the local file and previews the bands, as a check that the raster
matches the Earth Engine asset.

**What to check.** Band descriptions should name the crops in `C.CROP_NAMES` order. The CRS is
EPSG:4326 and the pixel is 0.0009 degrees, which is about 100 m at the equator but **not 1 ha
everywhere** — the cell shrinks in longitude as you move away from it. For any area calculation use the
shipped `*_pixel_area_ha_100m.tif`, which carries true geodesic hectares per pixel, and never
`count × 1 ha`.

NoData is 0 and the QML styles ship beside the GeoTIFFs, so QGIS opens them without a black background.

In [ ]:
import rasterio, numpy as np
f = f"{C.OUT_DIR}/{C.ISO2}_crop_fraction_{C.OUT_SCALE}m.tif"
if not os.path.exists(f):                       # cropland-only country (Djibouti)
    f = f"{C.OUT_DIR}/{C.ISO2}_crop_cropland_{C.OUT_SCALE}m.tif"
with rasterio.open(f) as r:
    print(f, "\n bands:", r.descriptions, "\n size:", r.shape, "\n crs:", r.crs, "\n pixel:", r.res)
    arr = r.read(out_shape=(r.count, r.height // 12, r.width // 12))
import matplotlib.pyplot as plt
names = C.CROP_NAMES if "fraction" in f else ["cropland (cl_pct)"]
fig, axs = plt.subplots(1, len(names), figsize=(4 * len(names), 4))
for ax, c, a in zip(np.atleast_1d(axs), names, arr):
    ax.imshow(np.ma.masked_equal(a, 0), cmap="viridis", vmin=0, vmax=30); ax.set_title(f"{c} % of cell"); ax.axis("off")
plt.show()

## 8 — Methodology documents
`<Country>/METHODOLOGY.md` and `.docx` carry the full method, results and references.

In [ ]:
from IPython.display import Markdown
doc = f"{C.COUNTRY_DIR}/METHODOLOGY.md"
if not os.path.exists(doc):
    doc = f"{C.COUNTRY_DIR}/README.md"      # Djibouti: cropland-only README
print(doc)
display(Markdown(open(doc).read()[:4000] + "\n\n*(truncated — open the .docx for the full document)*"))